# 04 Molecular Ratios

> Public source code; private observational inputs are not included. Outputs shown here were generated during the original thesis; the adapted notebooks have not been rerun against the raw data.


In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the project root from the working directory or an ancestor."""
    location = Path(start or Path.cwd()).resolve()
    for candidate in (location, *location.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory. The README.md and notebooks/ folder must exist.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
SPECTRA_DIR = PROJECT_ROOT / "spectra"
FIGURES_DIR = PROJECT_ROOT / "figures"
SAVE_FIGURES = False  # Figures render inline; optional exports disabled by default.
SAVE_TABLES = True    # Intermediate tables are required by downstream notebooks.
# Output folders are part of the repository layout; raw data remains private.


# Necesito calcular la masa molecular del CO, el ratio trazadores densos/CO y el ratio de gas termalizado R21/10

Cargo los paquetes y los archivos con los resultados del flujo en unidades de $I = \int T_\text{mb}\ dv$ ($\mathrm{K \ km \ s^{-1}}$) 

In [ ]:
import numpy as np
import os
import pandas as pd

results_dir = str(RESULTS_DIR)

# Cargar los datos de los archivos
co_ALMA = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCO/CO_ALMA_beam30m_Jy_K.dat'), sep="\t") #CO(1-0) con su beam y CO (2-1) con el suyo 
co10_ALMA_beam = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCO/CO10_ALMA_beam30m_Jy_K.dat'), sep="\t") # CO(1-0) con beam de CO(2-1) y de densa
co10_IRAM = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_CO10_moment_results.dat'), sep="\t") # CO(1-0) de class
co21_IRAM = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_CO21_moment_results.dat'), sep="\t") # CO(2-1) de class
co21_ALMA_beam10 = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCO/CO21_ALMA_beam30m_Jy_K.dat'), sep="\t") # CO(2-1) con beam de CO(1-0)
hcn = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HCN_moment_results.dat'), sep="\t") # HCN de class
hco = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HCO_moment_results.dat'), sep="\t") # HCO de class
hnc_ul = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosHNC/upper_limit_HNC_2_results.dat'), sep="\t") # HNC de class (solo límite superior) I < 3 * rms * \sqrt{\DeltaV * \deltaV} 
hnc = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HNC_moment_results.dat'), sep="\t") # HNC CEN de class (solo se detecta en el centro)


co10_ALMA = co_ALMA[["fuente", "I_CO10_ALMA_K", "err_I_CO10_ALMA_K"]] # CO(1-0) con su beam
co21_ALMA = co_ALMA[["fuente", "I_CO21_ALMA_K", "err_I_CO21_ALMA_K"]] # CO(2-1) con su beam
co10_beam21 = co10_ALMA_beam[["fuente", "I_CO10_21_ALMA_K", "err_I_CO10_21_ALMA_K"]] # CO(1-0) con beam de CO(2-1)
co10_beamdense = co10_ALMA_beam[["fuente", "I_CO10_DENSE_ALMA_K", "err_I_CO10_DENSE_ALMA_K"]] # CO(1-0) con beam de densa
co10_IRAM = co10_IRAM[["fuente", "I_manual", "SigmaI"]] # CO(1-0) de class
co21_IRAM = co21_IRAM[["fuente", "I_manual", "SigmaI"]] # CO(2-1) de class
co21_ALMA_beam10 = co21_ALMA_beam10[["fuente", "I_CO21_10_ALMA_K", "err_I_CO21_10_ALMA_K"]] # CO(2-1) con beam de CO(1-0)
hcn = hcn[["fuente", "I_manual", "SigmaI"]] # HCN de class
hco = hco[["fuente", "I_manual", "SigmaI"]] # HCO de class
hnc_ul = hnc_ul[["fuente", "I_ul_Jy", "I_ul_K"]] # HNC de class (solo límite superior) I < 3 * rms * \DeltaV
hnc = hnc[["fuente", "I_manual", "SigmaI"]] # HNC de class (solo se detecta en el centro)


# Ordenar por fuente para que coincidan las filas
co10_ALMA = co10_ALMA.sort_values("fuente").reset_index(drop=True)
co21_ALMA = co21_ALMA.sort_values("fuente").reset_index(drop=True)
co10_beam21 = co10_beam21.sort_values("fuente").reset_index(drop=True)
co10_beamdense = co10_beamdense.sort_values("fuente").reset_index(drop=True)
co10_IRAM = co10_IRAM.sort_values("fuente").reset_index(drop=True)
co21_IRAM = co21_IRAM.sort_values("fuente").reset_index(drop=True)
co21_ALMA_beam10 = co21_ALMA_beam10.sort_values("fuente").reset_index(drop=True)
hcn = hcn.sort_values("fuente").reset_index(drop=True)
hco = hco.sort_values("fuente").reset_index(drop=True)
hnc = hnc.sort_values("fuente").reset_index(drop=True)
hnc_ul = hnc_ul.sort_values("fuente").reset_index(drop=True)


## Ratio de gas termalizado $R_{21/10}$

Calculo $R_{21/10}=\frac{I_{CO(2-1)}}{I_{CO(1-0)}}$, dato que corresponde con el nivel de ionización del CO en la fuente. 

Calculo para cada fuente.

DEBO ASEGURARME DE UTILIZAR EL BEAM CORRECTO EN CADA CASO

In [ ]:
rows = []

# Primero con el CO(2-1) de IRAM y el CO(1-0) de ALMA con su beam. 

for i in range(len(co10_beam21)):
    fuente = co10_beam21["fuente"][i]

    # R21/10 usando I_manual
    R_2110_21IRAM     = co21_IRAM["I_manual"][i] / co10_beam21["I_CO10_21_ALMA_K"][i]
    err_R_2110_21IRAM = R_2110_21IRAM * np.sqrt((co21_IRAM["SigmaI"][i] / co21_IRAM["I_manual"][i])**2 +
                                                (co10_beam21["err_I_CO10_21_ALMA_K"][i] / co10_beam21["I_CO10_21_ALMA_K"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R2110_21IRAM"  : round(R_2110_21IRAM,     5),
        "err2110_21IRAM"  : round(err_R_2110_21IRAM, 5),
    })

df1= pd.DataFrame(rows)

# Luego con el CO(2-1) de ALMA con el beam de CO(1-0) y el CO(1-0) de IRAM. 

rows2 = []

for i in range (len(co10_IRAM)):
    fuente = co10_IRAM["fuente"][i]

    # R21/10 usando I_manual
    R_2110_21IRAM     = co21_ALMA_beam10["I_CO21_10_ALMA_K"][i] / co10_IRAM["I_manual"][i]
    err_R_2110_21IRAM = R_2110_21IRAM * np.sqrt((co21_ALMA_beam10["err_I_CO21_10_ALMA_K"][i] / co21_ALMA_beam10["I_CO21_10_ALMA_K"][i])**2 +
                                                (co10_IRAM["SigmaI"][i]  / co10_IRAM["I_manual"][i])**2)

    rows2.append({
        "fuente"   : fuente,
        "R2110_21ALMA"  : round(R_2110_21IRAM,     5),
        "err2110_21ALMA"  : round(err_R_2110_21IRAM, 5),
    })

df2 = pd.DataFrame(rows2)

out_df_CO = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_CO = out_df_CO.sort_values("fuente").reset_index(drop=True)
out_path = str(RESULTS_DIR / "W_RCO21_CO10.dat")
out_df_CO.to_csv(out_path, sep="\t", index=False)
print(out_df_CO.to_string(index=False))

## Ratio de HCN frente a CO (gas denso a difuso)

Ahora calculo el ratio entre flujo de HCN y CO, para comprobar si se trata de ULIRG o normal, además de la densidad de gas molecular denso

In [ ]:
rows = []
for i in range(len(co10_beamdense)):
    fuente = co10_beamdense["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HCN_CO     = hcn["I_manual"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i]
    err_R_HCN_CO = R_HCN_CO * np.sqrt((hcn["SigmaI"][i] / hcn["I_manual"][i])**2 +
                                       (co10_beamdense["err_I_CO10_DENSE_ALMA_K"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R_HCN_CO"  : round(R_HCN_CO,     5),
        "err_R_HCN_CO"  : round(err_R_HCN_CO, 5),
    })

order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HCN_CO = pd.DataFrame(rows)
out_df_HCN_CO["fuente"] = pd.Categorical(out_df_HCN_CO["fuente"], categories=order, ordered=True)
out_df_HCN_CO = out_df_HCN_CO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHCN_CO.dat")
out_df_HCN_CO.to_csv(out_path, sep="\t", index=False)
print(out_df_HCN_CO.to_string(index=False))

## Ratio de HCO frente a CO

Lo mismo con HCO+

In [ ]:
rows = []
for i in range(len(co10_beamdense)):
    fuente = co10_beamdense["fuente"][i]

    # R_HCO/CO10 usando I_manual
    R_HCO_CO     = hco["I_manual"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i]
    err_R_HCO_CO = R_HCO_CO * np.sqrt((hco["SigmaI"][i] / hco["I_manual"][i])**2 +
                                       (co10_beamdense["err_I_CO10_DENSE_ALMA_K"][i]  / co10_beamdense["I_CO10_DENSE_ALMA_K"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R_HCO_CO"  : round(R_HCO_CO,     5),
        "err_R_HCO_CO"  : round(err_R_HCO_CO, 5),
    })

order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HCO_CO = pd.DataFrame(rows)
out_df_HCO_CO["fuente"] = pd.Categorical(out_df_HCO_CO["fuente"], categories=order, ordered=True)
out_df_HCO_CO = out_df_HCO_CO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHCO_CO.dat")
out_df_HCO_CO.to_csv(out_path, sep="\t", index=False)
print(out_df_HCO_CO.to_string(index=False))

## Ratio de HNC frente a CO

Para HNC, al no detectarse de manera clara, se ha calculado un límite superior del flujo integrado. Se han utilizado 2 ecuaciones distintas: 

1. $I < 3 \ \text{rms} \ \Delta V$ 

2. $I_2 < 3 \ \text{rms} \ \sqrt{\Delta V \ \delta v}$ 

Se ha añadido el cálculo del ratio para el pointing central, ya que se detecta la línea. 

In [ ]:
rows = []
for i in range(len(co10_beamdense)):
    fuente = co10_beamdense["fuente"][i]

    # R_HNC/CO10 upper limit usando I_manual
    R_HNC_CO_ul     = hnc_ul["I_ul_K"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i]

    rows.append({
        "fuente"   : fuente,
        "R_HNC_CO_ul"  : round(R_HNC_CO_ul,     5),
    })

df1 = pd.DataFrame(rows)

rows2 = []
# R_HCN/CO10 usando I_manual solo para la detección del centro
R_HNC_CO     = hnc["I_manual"][0] / co10_beamdense["I_CO10_DENSE_ALMA_K"][0]
err_R_HNC_CO = R_HNC_CO * np.sqrt((hnc["SigmaI"][0] / hnc["I_manual"][0])**2 +
                                    (co10_beamdense["err_I_CO10_DENSE_ALMA_K"][0]  / co10_beamdense["I_CO10_DENSE_ALMA_K"][0])**2)

rows2.append({
    "fuente"   : co10_beamdense["fuente"][0],
    "R_HNC_CO"  : round(R_HNC_CO,     5),
    "err_R_HNC_CO"  : round(err_R_HNC_CO, 5),
})

df2 = pd.DataFrame(rows2)

out_df_HNC_CO = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HNC_CO["fuente"] = pd.Categorical(out_df_HNC_CO["fuente"], categories=order, ordered=True)
out_df_HNC_CO = out_df_HNC_CO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHNC_CO.dat")
out_df_HNC_CO.to_csv(out_path, sep="\t", index=False)
print(out_df_HNC_CO.to_string(index=False))

## Ratio de HCN frente a HCO+

Estos datos son todos de las observaciones de IRAM centradas en la misma frecuencia, por lo que la escala de Jy/beam es la misma

In [ ]:
rows = []
for i in range(len(hco)):
    fuente = hco["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HCN_HCO     = hco["I_manual"][i] / hcn["I_manual"][i]
    err_R_HCN_HCO = R_HCN_HCO * np.sqrt((hcn["SigmaI"][i] / hcn["I_manual"][i])**2 +
                                          (hco["SigmaI"][i]  / hco["I_manual"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R_HCN_HCO"  : round(R_HCN_HCO,     5),
        "err_R_HCN_HCO"  : round(err_R_HCN_HCO, 5),
    })

order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HCN_HCO = pd.DataFrame(rows)
out_df_HCN_HCO["fuente"] = pd.Categorical(out_df_HCN_HCO["fuente"], categories=order, ordered=True)
out_df_HCN_HCO = out_df_HCN_HCO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHCN_HCO.dat")
out_df_HCN_HCO.to_csv(out_path, sep="\t", index=False)
print(out_df_HCN_HCO.to_string(index=False))

## Ratio de HNC frente a HCO+

Los datos de HNC son límites superiores en Norte y Sur, estimados a partir de $3 \cdot \sigma_{rms} \cdot \sqrt{\Delta V \delta V}$.

In [ ]:
rows = []
for i in range(len(hco)):
    fuente = hco["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HNC_HCO_ul     = hnc_ul["I_ul_K"][i] / hco["I_manual"][i]

    rows.append({
        "fuente"   : fuente,
        "R_HNC_HCO_ul"  : round(R_HNC_HCO_ul,     5),
    })

df1 = pd.DataFrame(rows)

rows2 = []
# R_HCN/HCO usando I_manual solo para la detección del centro
R_HNC_HCO     = hnc["I_manual"][0] / hco["I_manual"][0]
err_R_HNC_HCO = R_HNC_HCO * np.sqrt((hnc["SigmaI"][0] / hnc["I_manual"][0])**2 +
                                       (hco["SigmaI"][0]  / hco["I_manual"][0])**2)

rows2.append({
    "fuente"   : hco["fuente"][0],
    "R_HNC_HCO"  : round(R_HNC_HCO,     5),
    "err_R_HNC_HCO"  : round(err_R_HNC_HCO, 5),
})

df2 = pd.DataFrame(rows2)

out_df_HNC_HCO = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HNC_HCO["fuente"] = pd.Categorical(out_df_HNC_HCO["fuente"], categories=order, ordered=True)
out_df_HNC_HCO = out_df_HNC_HCO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHNC_HCO.dat")
out_df_HNC_HCO.to_csv(out_path, sep="\t", index=False)
print(out_df_HNC_HCO.to_string(index=False))

## Ratio de HNC frente a HCN

Los datos de HNC son límites superiores en Norte y Sur

In [ ]:
rows = []
for i in range(len(hcn)):
    fuente = hcn["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HNC_HCN_ul     = hnc_ul["I_ul_K"][i] / hcn["I_manual"][i]

    rows.append({
        "fuente"   : fuente,
        "R_HNC_HCN_ul"  : round(R_HNC_HCN_ul,     5),
    })

df1 = pd.DataFrame(rows)

rows2 = []
# R_HNC/HCN usando I_manual solo para la detección del centro
R_HNC_HCN     = hnc["I_manual"][0] / hcn["I_manual"][0]
err_R_HNC_HCN = R_HNC_HCN * np.sqrt((hnc["SigmaI"][0] / hnc["I_manual"][0])**2 +
                                       (hcn["SigmaI"][0]  / hcn["I_manual"][0])**2)

rows2.append({
    "fuente"   : hcn["fuente"][0],
    "R_HNC_HCN"  : round(R_HNC_HCN,     5),
    "err_R_HNC_HCN"  : round(err_R_HNC_HCN, 5),
})

df2 = pd.DataFrame(rows2)

out_df_HNC_HCN = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HNC_HCN["fuente"] = pd.Categorical(out_df_HNC_HCN["fuente"], categories=order, ordered=True)
out_df_HNC_HCN = out_df_HNC_HCN.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHNC_HCN.dat")
out_df_HNC_HCN.to_csv(out_path, sep="\t", index=False)
print(out_df_HNC_HCN.to_string(index=False))

# IMPRIMO TABLA EN LATEX

In [ ]:
import re
import numpy as np
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP


def _round_half_up(x, ndigits=0):
    if pd.isna(x):
        return x
    q = Decimal("1").scaleb(-ndigits)
    return float(Decimal(str(x)).quantize(q, rounding=ROUND_HALF_UP))


def _format_detection(val, err, err_sigfigs=2, latex=True):
    if pd.isna(val) or pd.isna(err):
        return None

    err = abs(float(err))
    val = float(val)

    if err == 0:
        return f"${val} \\pm 0$" if latex else f"{val} ± 0"

    exponent = int(np.floor(np.log10(err)))
    decimals = err_sigfigs - 1 - exponent

    err_r = _round_half_up(err, decimals)
    val_r = _round_half_up(val, decimals)

    if decimals > 0:
        val_str = f"{val_r:.{decimals}f}"
        err_str = f"{err_r:.{decimals}f}"
    else:
        val_str = f"{val_r:.0f}"
        err_str = f"{err_r:.0f}"

    return f"${val_str} \\pm {err_str}$" if latex else f"{val_str} ± {err_str}"


def _format_upper_limit(val, err_sigfigs=2, latex=True):
    if pd.isna(val):
        return None

    val = abs(float(val))
    if val == 0:
        return "$< 0$" if latex else "< 0"

    exponent = int(np.floor(np.log10(val)))
    decimals = err_sigfigs - 1 - exponent
    val_r = _round_half_up(val, decimals)

    if decimals > 0:
        val_str = f"{val_r:.{decimals}f}"
    else:
        val_str = f"{val_r:.0f}"

    return f"$< {val_str}$" if latex else f"< {val_str}"


def _base_name_from_err(err_col):
    if err_col.startswith("err_"):
        return err_col[4:]
    if err_col.startswith("err"):
        return "R" + err_col[3:]
    return None


def _choose_output_name(base, custom_map=None):
    if custom_map and base in custom_map:
        return custom_map[base]
    return base


def combinar_ratios_robusto(
    dfs,
    order=None,
    latex=True,
    err_sigfigs=2,
    custom_map=None,
    keep_fuente=True,
    fuente_map=None
):
    """
    Combina varios DataFrames de ratios en un único DataFrame formateado.
    Soporta:
    - múltiples detecciones en un mismo df
    - upper limits _ul, _ul_2, ...
    - prioridad detección > upper limit
    - renombrado de fuentes (NOR/CEN/SOU -> North/Center/South)
    """
    if order is None:
        order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]

    if fuente_map is None:
        fuente_map = {
            "NGC3110_NOR": "North",
            "NGC3110_CEN": "Center",
            "NGC3110_SOU": "South",
        }

    bloques = []

    for df_idx, df in enumerate(dfs):
        if "fuente" not in df.columns:
            raise ValueError(f"El DataFrame {df_idx} no tiene columna 'fuente'.")

        df = df.copy()
        df["fuente"] = pd.Categorical(df["fuente"], categories=order, ordered=True)
        df = df.sort_values("fuente").reset_index(drop=True)

        cols = [c for c in df.columns if c != "fuente"]

        detections = {}
        for col in cols:
            if col.startswith("err"):
                base = _base_name_from_err(col)
                if base in df.columns:
                    detections[base] = col

        ul_groups = {}
        ul_pattern = re.compile(r"^(.*)_ul(?:_(\d+))?$")

        for col in cols:
            m = ul_pattern.match(col)
            if m:
                base = m.group(1)
                idx = m.group(2)
                idx = int(idx) if idx is not None else 1
                ul_groups.setdefault(base, []).append((idx, col))

        for base in ul_groups:
            ul_groups[base] = [c for _, c in sorted(ul_groups[base], key=lambda x: x[0])]

        out = pd.DataFrame({"fuente": df["fuente"].astype(str)})

        for base, err_col in detections.items():
            out_name = _choose_output_name(base, custom_map)
            out[out_name] = df.apply(
                lambda row: _format_detection(
                    row[base], row[err_col],
                    err_sigfigs=err_sigfigs,
                    latex=latex
                ),
                axis=1
            )

        for base, ul_cols in ul_groups.items():
            if base in detections:
                out_name = _choose_output_name(base, custom_map)

                def fill_det_or_ul(row):
                    det_txt = _format_detection(
                        row[base], row[detections[base]],
                        err_sigfigs=err_sigfigs,
                        latex=latex
                    )
                    if det_txt is not None:
                        return det_txt

                    for ul_col in ul_cols:
                        ul_txt = _format_upper_limit(
                            row[ul_col],
                            err_sigfigs=err_sigfigs,
                            latex=latex
                        )
                        if ul_txt is not None:
                            return ul_txt

                    return r"\ldots" if latex else "..."

                out[out_name] = df.apply(fill_det_or_ul, axis=1)

            else:
                if len(ul_cols) == 1:
                    out_name = _choose_output_name(ul_cols[0], custom_map)
                    out[out_name] = df[ul_cols[0]].apply(
                        lambda x: _format_upper_limit(x, err_sigfigs=err_sigfigs, latex=latex)
                        or (r"\ldots" if latex else "...")
                    )
                else:
                    for ul_col in ul_cols:
                        out_name = _choose_output_name(ul_col, custom_map)
                        out[out_name] = df[ul_col].apply(
                            lambda x: _format_upper_limit(x, err_sigfigs=err_sigfigs, latex=latex)
                            or (r"\ldots" if latex else "...")
                        )

        bloques.append(out)

    final_df = bloques[0]
    for b in bloques[1:]:
        final_df = pd.merge(final_df, b, on="fuente", how="outer")

    final_df["fuente"] = pd.Categorical(final_df["fuente"], categories=order, ordered=True)
    final_df = final_df.sort_values("fuente").reset_index(drop=True)

    # renombrar fuentes
    final_df["fuente"] = final_df["fuente"].map(fuente_map)

    if not keep_fuente:
        final_df = final_df.drop(columns="fuente")

    return final_df


def latex_table(headers, headers2, data, caption=None, label=None,
                table_env="table*", position="ht"):
    """
    Genera una tabla LaTeX ajustada al ancho de página.

    headers  : incluye 'Pointing' como primera celda
    headers2 : NO incluye 'Pointing'
    data     : filas completas (incluyendo Pointing como primera columna)
    """

    ncols = 1 + len(headers2)
    align = "l" + "c" * (ncols - 1)

    lines = []

    if caption or label:
        lines.append(f"\\begin{{{table_env}}}[{position}]")
        lines.append("\\centering")

    lines.append("\\resizebox{\\textwidth}{!}{%")
    lines.append(f"\\begin{{tabular}}{{{align}}}")
    lines.append("\\hline")

    lines.append(" & ".join(headers) + r" \\")
    lines.append(" & ".join([""] + headers2) + r" \\")
    lines.append("\\hline")

    for row in data:
        if len(row) != ncols:
            raise ValueError(f"Fila con {len(row)} columnas, esperado {ncols}: {row}")
        lines.append(" & ".join(map(str, row)) + r" /\\".replace("/", ""))

    lines.append("\\hline")
    lines.append("\\end{tabular}%")
    lines.append("}")

    if caption:
        lines.append(f"\\caption{{{caption}}}")
    if label:
        lines.append(f"\\label{{{label}}}")

    if caption or label:
        lines.append(f"\\end{{{table_env}}}")

    return "\n".join(lines)


def preparar_tabla_latex_desde_dfs(
    dfs,
    headers,
    headers2,
    order=None,
    custom_map=None,
    fuente_map=None,
    caption=None,
    label=None,
    table_env="table*",
    position="ht"
):
    """
    Combina DataFrames de ratios y devuelve:
    - latex_code: bloque exacto para copiar y pegar
    - df_final: DataFrame final ya ordenado
    """

    df_final = combinar_ratios_robusto(
        dfs=dfs,
        order=order,
        latex=True,
        custom_map=custom_map,
        fuente_map=fuente_map
    )

    # La primera columna sigue siendo 'fuente' internamente
    df_final = df_final.rename(columns={"fuente": "Pointing"})

    # Ahora Pointing NO está en headers2, así que hay que añadirlo aparte
    expected_cols = ["Pointing"] + headers2

    missing = [c for c in expected_cols if c not in df_final.columns]
    if missing:
        raise ValueError(f"Faltan estas columnas en df_final: {missing}")

    df_final = df_final[expected_cols]
    data = df_final.values.tolist()

    latex_code = latex_table(
        headers=headers,
        headers2=headers2,
        data=data,
        caption=caption,
        label=label,
        table_env=table_env,
        position=position
    )

    return latex_code, df_final


headers = [
    "Pointing",
    "\\multicolumn{2}{c}{CO (2--1)/CO (1--0)}",
    '\\multicolumn{2}{c}{HCN (1--0)}',
    '\\multicolumn{1}{c}{HCO$^+$ (1--0)}',
    '\\multicolumn{3}{c}{HNC (1--0)}'
]

headers2 = [
    'IRAM 10.85"',
    'IRAM 20.70"',
    "HCN/CO",
    "HCN/HCO$^+$",
    "HCO$^+$/CO",
    "HNC/CO",
    "HNC/HCO$^+$",
    "HNC/HCN"
]

custom_map = {
    "R2110_21ALMA": 'IRAM 10.85"',
    "R2110_21IRAM": 'IRAM 20.70"',
    "R_HCN_CO": "HCN/CO",
    "R_HCN_HCO": "HCN/HCO$^+$",
    "R_HCO_CO": "HCO$^+$/CO",
    "R_HNC_CO": "HNC/CO",
    "R_HNC_HCO": "HNC/HCO$^+$",
    "R_HNC_HCN": "HNC/HCN"
}

fuente_map = {
    "NGC3110_NOR": "North",
    "NGC3110_CEN": "Center",
    "NGC3110_SOU": "South"
}

latex_code, df_final = preparar_tabla_latex_desde_dfs(
    dfs=[
        out_df_CO,
        out_df_HCN_CO,
        out_df_HCN_HCO,
        out_df_HCO_CO,
        out_df_HNC_CO,
        out_df_HNC_HCO,
        out_df_HNC_HCN
    ],
    headers=headers,
    headers2=headers2,
    order=["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"],
    custom_map=custom_map,
    fuente_map=fuente_map,
    caption="Molecular line ratios in NGC~3110.",
    label="tab:ratios_ngc3110",
    table_env="table*",
    position="ht"
)

print(latex_code)